In [ ]:
import numpy as np
from learn_s_hat_toy import make_srs
from gould_2026.plotting import paper_plot_context, Palette
from gould_2026.datasets import ArrayWithTime
import scipy.stats
from io import StringIO
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
output_1_learn_toy_manifold = None
output_2_stat_text = None

In [ ]:
output_2_stat_text = Path(output_2_stat_text)

In [ ]:
srs = make_srs(np.random.default_rng(2), n_runs=1, show_tqdm=False, add_s_hat_error_function=True, u_function='curvy')

In [ ]:
fig, ax = plt.subplots()
colors = [Palette.stim_regressed, Palette.blind]

shes = []
for i, k in enumerate(['learning from stim', 'unaware of stim']):
    for sr in srs[k]:
        she = ArrayWithTime.from_list(sr.log['s_hat_error'])
        stim_samples = sr.log['stim_intended_samples']
        she, _ = ArrayWithTime.align_indices(she, stim_samples)
        she = np.mean(she**2, axis=2)
        ax.plot(she.mean(axis=1), label=k, color=colors[i])
        shes.append(she)


ax.set_xlim([0, 50])
ax.set_xlabel('# of stimuli')
ax.set_ylabel(r'~$\mathbb{E}\Vert \hat S - S \Vert^2$')
ax.set_ylim(bottom=0)
# ax.legend()


if output_1_learn_toy_manifold is not None:
    fig.savefig(output_1_learn_toy_manifold)


In [ ]:
shes = np.array(shes)
test_sample = 9
aware, unaware = shes[:, test_sample,]
test_result = scipy.stats.wilcoxon(unaware, aware)
test_string = StringIO()
test_string.write(f"'learning from stim' vs 'unaware of stim' at {test_sample = }\n")
test_string.write(f"{test_result = }\n")
test_string.write(f"Δ = {np.mean(unaware) - np.mean(aware):.3f} {np.mean(unaware)=:.3f} {np.mean(aware)=:.3f}\n")

if output_2_stat_text is not None:
    with output_2_stat_text.open('w') as fhan:
        fhan.write(test_string.getvalue())
